# MLOps II - Clase 1: Flujo de Datos con APIs REST


Bienvenidos a la primera clase práctica de MLOps II. En esta sesión construiremos de forma **progresiva** las piezas fundamentales para desplegar modelos de Machine Learning en producción.

**Objetivos de la Clase:**
1. Consumo de datos externos.
2. Creación de APIs: De básicas a compatibles con Web (CORS).
3. Evolución del Despliegue de Modelos: Desde un script simple hasta una API robusta con validación, seguridad y versionado.

## Parte 1: Consumo Resiliente de Datos a través de APIs
En MLOps, los pipelines dependen de datos externos. ¿Qué pasa si la API externa se cae o tarda mucho? 
Utilizamos **Timeouts** y **Manejo de Excepciones** para no bloquear nuestros sistemas.

In [38]:
import requests
from requests.exceptions import HTTPError, Timeout
import json

url = "https://jsonplaceholder.typicode.com/users"

print("Iniciando consumo de API externa...")
try:
    # timeout=5 significa: 'si en 5 segundos no hay respuesta, aborta'
    response = requests.get(url, timeout=5)
    response.raise_for_status() # Lanza error si el status NO es 200-299
    
    datos_usuarios = response.json()
    print(f"¡Éxito! Se obtuvieron {len(datos_usuarios)} registros.")
    
    # Guardamos el dataset inicial
    with open("datos_locales.json", "w", encoding="utf-8") as f:
        json.dump(datos_usuarios, f, indent=4)
        
except Timeout:
    print("Error Crítico: El servidor tardó demasiado (Timeout).")
except HTTPError as http_err:
    print(f"Error HTTP: {http_err}")
except Exception as err:
    print(f"Error inesperado: {err}")

Iniciando consumo de API externa...
¡Éxito! Se obtuvieron 10 registros.


## Parte 2: Creación de una API y el problema del CORS

### 2.1 API Convencional
Comenzaremos creando una API muy sencilla que lee nuestro archivo local y lo expone en la red.

In [39]:
%%writefile simple_api.py
from fastapi import FastAPI
import json

app = FastAPI(title="API Básica")

@app.get("/usuarios")
def obtener_usuarios():
    try:
        with open("datos_locales.json", "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        return {"error": "Archivo no encontrado."}


Overwriting simple_api.py


> **Levantar la API:** Abre una terminal y ejecuta: `python -m uvicorn simple_api:app --reload --port 8001`

**¿Cómo probamos esta API?**
Si la probamos desde otro script de Python (Backend-to-Backend), ¡funcionará perfectamente! Python no tiene las restricciones de seguridad que tiene un navegador web. Ejecuta la celda de abajo para comprobarlo:

In [41]:
# PRUEBA DESDE PYTHON (BACKEND)
import requests
try:
    res = requests.get("http://localhost:8001/usuarios")
    print(f"Prueba Backend exitosa. Se obtuvieron {len(res.json())} registros.")
    datos_usuarios = res.json()
    info_usuario = datos_usuarios[0].keys()
    print(f"Campos disponibles: {info_usuario}")
    for usuario in datos_usuarios:
        print(f"ID: {usuario['id']}, Nombre: {usuario['name']}")

except requests.exceptions.ConnectionError:
    print("Error: ¿Olvidaste levantar la API en la terminal con el puerto 8001?")

Prueba Backend exitosa. Se obtuvieron 10 registros.
Campos disponibles: dict_keys(['id', 'name', 'username', 'email', 'address', 'phone', 'website', 'company'])
ID: 1, Nombre: Leanne Graham
ID: 2, Nombre: Ervin Howell
ID: 3, Nombre: Clementine Bauch
ID: 4, Nombre: Patricia Lebsack
ID: 5, Nombre: Chelsey Dietrich
ID: 6, Nombre: Mrs. Dennis Schulist
ID: 7, Nombre: Kurtis Weissnat
ID: 8, Nombre: Nicholas Runolfsdottir V
ID: 9, Nombre: Glenna Reichert
ID: 10, Nombre: Clementina DuBuque


**La Pesadilla del Frontend (Same-Origin Policy)**
¿Qué pasa si un Frontend (una página web en Angular, React o HTML puro) intenta consumir esos mismos datos? El navegador lo bloqueará por la política del "Mismo Origen".
Vamos a simular un Frontend creando un archivo HTML simple.

In [42]:
%%writefile frontend_API.html
<!DOCTYPE html>
<html><body>
    <h2>Prueba Frontend a API (Puerto 8001)</h2>
    <button onclick="testAPI()">Consultar Datos</button>
    <p id="resultado">Esperando...</p>
    <script>
    function testAPI() {
        fetch('http://localhost:8001/usuarios')
        .then(response => response.json())
        .then(data => document.getElementById('resultado').innerHTML = "<span style='color:green'>¡Exito! Datos cargados.</span>")
        .catch(error => {
            document.getElementById('resultado').innerHTML = "<strong style='color:red'>ERROR: " + error.message + "</strong><br><em>Presiona F12 y ve a la pestaña 'Console' para ver el bloqueo por política CORS.</em>";
            console.error("¡LA PETICIÓN FUE BLOQUEADA POR CORS!", error);
        });
    }
    </script>
</body></html>

Writing frontend_API.html


> **Instrucciones para el Alumno:** 
> 1. Ve a la carpeta donde está este notebook.
> 2. Haz doble clic en `frontend_API.html` para abrirlo en tu navegador.
> 3. Haz clic en el botón. Verás que falla estrepitosamente. Abre las herramientas de desarrollador (F12 -> Consola) para ver el error oficial de CORS (Cross-Origin Resource Sharing).

---

### 2.2 La Solución: API con CORSMiddleware
Para solucionar esto, incluimos el `CORSMiddleware`. Este middleware le dice al navegador web: *"Tranquilo, conozco a este usuario, permítele consumir mis datos aunque venga de otra página web".*

In [43]:
%%writefile api_cors.py
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
import json

app = FastAPI(title="API básica con CORS")

# Configuramos el Middleware para permitir conexiones externas
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], # En producción, reemplazar '*' por los dominios de tu Frontend
    allow_credentials=True,
    allow_methods=["GET", "POST"],
    allow_headers=["*"],
)

@app.get("/usuarios")
def obtener_usuarios():
    try:
        with open("datos_locales.json", "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        return {"error": "Archivo no encontrado."}


Writing api_cors.py


> **Levantar la API 2:** Abre OTRA terminal (o cancela la anterior) y ejecuta: `python -m uvicorn api_cors:app --reload --port 8002`

Ahora generaremos un nuevo Frontend apuntando a esta nueva API sana en el puerto 8002.

In [44]:
%%writefile frontend_API_cors.html
<!DOCTYPE html>
<html><body>
    <h2>Prueba Frontend a API CON CORS (Puerto 8002)</h2>
    <button onclick="testAPI()">Consultar Datos</button>
    <p id="resultado">Esperando...</p>
    <script>
    function testAPI() {
        fetch('http://localhost:8002/usuarios')
        .then(response => response.json())
        .then(data => {
            console.log("¡Éxito! El middleware CORS permitió la conexión.");console.log("Datos recibidos:", data);
            document.getElementById('resultado').innerHTML = "<span style='color:green'>¡Éxito total! Se cargaron " + data.length + " usuarios en el frontend. El middleware CORS hizo su trabajo.</span>";
            })
        .catch(error => {
            document.getElementById('resultado').innerHTML = "<strong style='color:red'>ERROR: " + error.message + "</strong>";
        });
    }
    </script>
</body></html>

Writing frontend_API_cors.html


> **Prueba Final:** Abre `frontend_API_cors.html` en tu navegador y haz clic. ¡Verás que los datos ahora sí fluyen perfectamente hacia el frontend!

---

## Parte 3: Evolución de una API de Machine Learning
Desplegar un modelo no es solo hacer un endpoint. Veremos 3 niveles de madurez arquitectónica.

### Nivel 1: La API Sencilla e Ingenua
Recibe parámetros simples en la URL (Query Parameters) y devuelve una predicción binaria (0 o 1). Es propenso a errores porque no valida la estructura compleja de los datos.

In [45]:
%%writefile api_ml_nivel1.py
from fastapi import FastAPI

app = FastAPI(title="ML API - Nivel 1")

def modelo_dummy(f1: float, f2: float) -> int:
    # Si la suma ponderada supera el umbral, clase 1 (Positivo), si no, 0 (Negativo)
    return 1 if (f1 * 0.5 + f2) > 10 else 0

@app.post("/predecir_basico")
def predecir_simple(feature_1: float, feature_2: float):
    prediccion = modelo_dummy(feature_1, feature_2)
    return {"prediccion": prediccion}


Overwriting api_ml_nivel1.py


> **Ejecución:** `python -m uvicorn api_ml_nivel1:app --reload --port 8003`

Vamos a hacer pruebas ahora desde Python:


In [54]:
import requests

URL_BASE = "http://localhost:8003"  # 

print("--- Probando Modelo Nivel 1 ---")
resp_v1 = requests.post(
    f"{URL_BASE}/predecir_basico?feature_1=2&feature_2=5"
)
print(f"Status: {resp_v1.status_code} | Respuesta: {resp_v1.text}")

--- Probando Modelo V1 ---
Status: 200 | Respuesta: {"prediccion":0}




### Nivel 2: Especificando Características con Pydantic
Los modelos reales reciben decenas de variables. Pasarlas por URL es imposible. Usaremos **Pydantic** para forzar que los datos vengan en el *Body* del request con un esquema estricto (JSON).

In [56]:
%%writefile api_ml_nivel2.py
from fastapi import FastAPI
from pydantic import BaseModel, Field

app = FastAPI(title="ML API - Nivel 2 (Con Pydantic)")

# 1. ESQUEMA DE ENTRADA ESTRICTO
class Caracteristicas(BaseModel):
    feature_1: float = Field(..., description="Medida en cm", example=12.5)
    feature_2: float = Field(..., description="Peso en kg", example=4.2)

# 2. ESQUEMA DE SALIDA
class Resultado(BaseModel):
    clase_predicha: int
    etiqueta: str

def modelo_dummy(f1: float, f2: float) -> int:
    return 1 if (f1 * 0.5 + f2) > 10 else 0

@app.post("/predecir_estructurado", response_model=Resultado)
def predecir_mejorado(datos: Caracteristicas):
    pred = modelo_dummy(datos.feature_1, datos.feature_2)
    etiqueta = "Aprobado" if pred == 1 else "Rechazado"
    
    return Resultado(clase_predicha=pred, etiqueta=etiqueta)


Overwriting api_ml_nivel2.py


> **Ejecución:** `python -m uvicorn api_ml_nivel2:app --reload --port 8004`


In [60]:
import requests

URL_BASE = "http://localhost:8004"
PAYLOAD = {"feature_1": 15.0, "feature_2": 5.0}


print("--- Probando Modelo Nivel 2 ---")

resp_v1 = requests.post(f"{URL_BASE}/predecir_estructurado", json=PAYLOAD)
print(f"Status: {resp_v1.status_code} | Respuesta: {resp_v1.json()}\n")





--- Probando Modelo Nivel 2 ---
Status: 200 | Respuesta: {'clase_predicha': 1, 'etiqueta': 'Aprobado'}



---

### Nivel 3: API Avanzada (Seguridad y Versionado de Modelos)
La versión final para producción. Agregamos:
1. **Seguridad (API Key)** para evitar abusos del endpoint.
2. **Versionado de Modelos** (`v1` y `v2`) para permitir actualizaciones en caliente.

In [61]:
%%writefile api_ml_nivel3.py
from fastapi import FastAPI, HTTPException, Security, APIRouter
from fastapi.security.api_key import APIKeyHeader
from pydantic import BaseModel, Field

app = FastAPI(title="ML API - Nivel 3 (Producción)")

# --- SEGURIDAD ---
api_key_header = APIKeyHeader(name="X-API-KEY", auto_error=False)
CLIENTES_AUTORIZADOS = {"token-secreto-123": "Cliente_Alpha"}

def validar_token(api_key: str = Security(api_key_header)):
    if api_key not in CLIENTES_AUTORIZADOS:
        raise HTTPException(status_code=403, detail="Token inválido.")
    return CLIENTES_AUTORIZADOS[api_key]

# --- ESQUEMAS ---
class Caracteristicas(BaseModel):
    feature_1: float = Field(..., example=12.5)
    feature_2: float = Field(..., example=4.2)

# --- REGISTRO DE MODELOS (Simulado) ---
class ModeloV1:
    def predecir(self, f1: float, f2: float) -> int:
        return 1 if (f1 * 0.5 + f2) > 10 else 0  # Lógica antigua

class ModeloV2:
    def predecir(self, f1: float, f2: float) -> int:
        return 1 if (f1 * 0.8 + f2 ** 1.2) > 15 else 0 # Nueva lógica optimizada

modelos = {"v1": ModeloV1(), "v2": ModeloV2()}

# --- ENRUTAMIENTO VERSIONADO ---
router_v1 = APIRouter(prefix="/v1", tags=["Modelo V1"])
router_v2 = APIRouter(prefix="/v2", tags=["Modelo V2 (Beta)"])

@router_v1.post("/predecir")
def prediccion_v1(datos: Caracteristicas, cliente: str = Security(validar_token)):
    res = modelos["v1"].predecir(datos.feature_1, datos.feature_2)
    return {"version": "1.0", "prediccion": res}

@router_v2.post("/predecir")
def prediccion_v2(datos: Caracteristicas, cliente: str = Security(validar_token)):
    res = modelos["v2"].predecir(datos.feature_1, datos.feature_2)
    return {"version": "2.0", "prediccion": res}

app.include_router(router_v1)
app.include_router(router_v2)


Writing api_ml_nivel3.py


> **Ejecución Final:** `python -m uvicorn api_ml_nivel3:app --reload --port 8080`
> 
> *Nota: Levanta esta última API en la terminal y ejecuta la siguiente celda para probarla.*

## Parte 4: Consumiendo la API de Producción (Testing)

In [63]:
import requests

URL_BASE = "http://localhost:8080"
PAYLOAD = {"feature_1": 15.0, "feature_2": 5.0}
HEADERS = {"X-API-KEY": "token-secreto-123"}
HEADERS_INVALID = {"X-API-KEY": "token-equivocado"}  # token inválido

print("--- Probando Modelo V1 ---")
resp_v1 = requests.post(f"{URL_BASE}/v1/predecir", json=PAYLOAD, headers=HEADERS)
print(f"Status: {resp_v1.status_code} | Respuesta: {resp_v1.json()}\n")

print("--- Probando Modelo V2 (Mismos datos, diferente lógica) ---")
resp_v2 = requests.post(f"{URL_BASE}/v2/predecir", json=PAYLOAD, headers=HEADERS)
print(f"Status: {resp_v2.status_code} | Respuesta: {resp_v2.json()}\n")

print("--- Probando con token inválido ---")
resp_invalid = requests.post(f"{URL_BASE}/v1/predecir", json=PAYLOAD, headers=HEADERS_INVALID)
print(f"Status: {resp_invalid.status_code} | Respuesta: {resp_invalid.json()}")

--- Probando Modelo V1 ---
Status: 200 | Respuesta: {'version': '1.0', 'prediccion': 1}

--- Probando Modelo V2 (Mismos datos, diferente lógica) ---
Status: 200 | Respuesta: {'version': '2.0', 'prediccion': 1}

--- Probando con token inválido ---
Status: 403 | Respuesta: {'detail': 'Token inválido.'}
